# Stage 5.1: Predictor-domain contribution

Stage 5.1 examines how outer-validation performance changes when predefined predictor domains are omitted from the full 69-predictor set or added separately to a common 23-predictor baseline.

The same five outer folds are used for MLR, RF, XGBoost and supplementary BRF. Within each model and outer fold, the Stage 5-selected hyperparameters and imbalance setting are held fixed while the predictor specification changes. Preprocessing is fitted on the corresponding outer-training sample.

The held-out test sample is not used. The comparisons are conditional predictive analyses under fixed model settings; they are not causal effects or model-independent rankings of domain importance.

## Part 1: Inputs and training sample

In [1]:
# 1: Import packages and set project paths

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "Stage 5.1 requires xgboost in the project environment."
    ) from exc

try:
    from imblearn.ensemble import BalancedRandomForestClassifier
except ImportError as exc:
    raise ImportError(
        "Stage 5.1 requires imbalanced-learn in the project environment."
    ) from exc

working_directory = Path.cwd().resolve()

# Locate the project root from the Stage 3 and Stage 5 files required here.
project_root = next(
    (
        directory
        for directory in [working_directory, *working_directory.parents]
        if (
            directory
            / "data_derived"
            / "stage_3_final_modelling_dataset"
            / "final_modelling_dataset.csv"
        ).is_file()
        and (
            directory
            / "data_derived"
            / "stage_5_nested_model_development"
            / "stage_5_outer_selection_register.csv"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "The Stage 3 modelling dataset and Stage 5 model-development files could not be located."
    )

data_derived = project_root / "data_derived"

stage_2_directory = data_derived / "stage_2_predictor_construction"
stage_3_directory = data_derived / "stage_3_final_modelling_dataset"
stage_4_directory = data_derived / "stage_4_modelling_design"
stage_5_directory = data_derived / "stage_5_nested_model_development"
stage_5_1_directory = data_derived / "stage_5_1_predictor_domain_contribution"
stage_5_1_directory.mkdir(parents=True, exist_ok=True)

paths = {
    "modelling data": stage_3_directory / "final_modelling_dataset.csv",
    "predictor manifest": stage_2_directory / "stage_2_final_predictor_manifest.csv",
    "split": stage_4_directory / "stage_4_train_test_split.csv",
    "outer folds": stage_4_directory / "stage_4_outer_folds.csv",
    "predictor roles": stage_4_directory / "stage_4_predictor_preprocessing_roles.csv",
    "Stage 4 domain omission": stage_4_directory / "stage_4_domain_omission_specification.csv",
    "Stage 4 domain addition": stage_4_directory / "stage_4_domain_addition_specification.csv",
    "Stage 4 audit": stage_4_directory / "stage_4_design_audit.csv",
    "Stage 5 search specification": stage_5_directory / "stage_5_search_specification.csv",
    "Stage 5 selections": stage_5_directory / "stage_5_outer_selection_register.csv",
    "Stage 5 outer predictions": stage_5_directory / "stage_5_selected_outer_predictions.csv",
    "Stage 5 audit": stage_5_directory / "stage_5_final_audit.csv",
}

missing = [name for name, path in paths.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing required input file(s): " + ", ".join(missing)
    )

print("Project root located.")
print(
    "Stage 5.1 output directory: "
    f"{stage_5_1_directory.relative_to(project_root).as_posix()}"
)

Project root located.
Stage 5.1 output directory: data_derived/stage_5_1_predictor_domain_contribution


In [2]:
# 2: Load the fixed Stage 4 and Stage 5 inputs

split_assignment = pd.read_csv(
    paths["split"],
    dtype={"NSID": "string"},
)
outer_assignments = pd.read_csv(
    paths["outer folds"],
    dtype={"NSID": "string"},
)
predictor_roles = pd.read_csv(paths["predictor roles"])
predictor_manifest = pd.read_csv(paths["predictor manifest"])

domain_omission_spec = pd.read_csv(paths["Stage 4 domain omission"])
domain_addition_spec = pd.read_csv(paths["Stage 4 domain addition"])
stage_4_audit = pd.read_csv(paths["Stage 4 audit"])

stage_5_search = pd.read_csv(
    paths["Stage 5 search specification"],
    keep_default_na=False,
    na_values=[""],
)
stage_5_selections = pd.read_csv(
    paths["Stage 5 selections"],
    keep_default_na=False,
    na_values=[""],
)
stage_5_predictions = pd.read_csv(
    paths["Stage 5 outer predictions"],
    dtype={"NSID": "string"},
    keep_default_na=False,
    na_values=[""],
)
stage_5_audit = pd.read_csv(paths["Stage 5 audit"])

training_ids = set(
    split_assignment.loc[
        split_assignment["sample"].eq("Training"),
        "NSID",
    ]
)
test_ids = set(
    split_assignment.loc[
        split_assignment["sample"].eq("Test"),
        "NSID",
    ]
)

modelling_data = pd.read_csv(
    paths["modelling data"],
    dtype={"NSID": "string"},
)
training_data = (
    modelling_data.loc[
        modelling_data["NSID"].isin(training_ids)
    ]
    .copy()
    .reset_index(drop=True)
)
del modelling_data

predictor_columns = predictor_roles["Predictor"].tolist()

# Predictor names and domain assignments must agree across the two Stage 2 registers.
assert predictor_manifest["Predictor"].is_unique
assert set(predictor_manifest["Predictor"]) == set(predictor_columns)

print(f"Training participants: {len(training_data):,}")
print(f"Predictors: {len(predictor_columns)}")
print(
    "Represented domains: "
    f"{predictor_manifest['Predictor domain'].nunique()}"
)

Training participants: 7,619
Predictors: 69
Represented domains: 10


In [3]:
# 3: Audit the Stage 5.1 inputs

stage_5_1_models = ["MLR", "RF", "XGBoost", "BRF"]

assert not stage_5_selections.duplicated(["Model", "Outer fold"]).any()

selection_counts = (
    stage_5_selections.loc[
        stage_5_selections["Model"].isin(stage_5_1_models)
    ]
    .groupby("Model")["Outer fold"]
    .nunique()
)

prediction_counts = (
    stage_5_predictions.loc[
        stage_5_predictions["Model"].isin(stage_5_1_models)
    ]
    .groupby("Model")["NSID"]
    .nunique()
)

addition_reference_rows = domain_addition_spec[
    domain_addition_spec["Analysis"].eq("Common-baseline reference")
]
addition_rows = domain_addition_spec[
    domain_addition_spec["Analysis"].eq("Separate domain addition")
]

input_checks = {
    "Stage 4 audit passed": stage_4_audit["Passed"].all(),
    "Stage 5 audit passed": stage_5_audit["Passed"].all(),
    "Training sample contains 7,619 participants": len(training_data) == 7_619,
    "Training identifiers are unique": training_data["NSID"].is_unique,
    "Training and test identifiers do not overlap": training_ids.isdisjoint(test_ids),
    "Training data contain no test identifiers": set(training_data["NSID"]).isdisjoint(test_ids),
    "Outer folds contain exactly the training identifiers": set(outer_assignments["NSID"]) == training_ids,
    "All 69 predictors are registered": len(predictor_columns) == 69
        and predictor_roles["Predictor"].is_unique,
    "All 10 represented domains are available": predictor_manifest["Predictor domain"].nunique() == 10,
    "LODO specification contains 10 domains": len(domain_omission_spec) == 10,
    "Addition specification has one common baseline": len(addition_reference_rows) == 1,
    "Addition specification has six separate additions": len(addition_rows) == 6,
    "Each Stage 5.1 model has five Stage 5 selections": selection_counts.reindex(stage_5_1_models).eq(5).all(),
    "Each Stage 5.1 model has 7,619 Stage 5 outer predictions": prediction_counts.reindex(stage_5_1_models).eq(7_619).all(),
    "Stage 5.1 model predictions contain no test identifiers": set(
        stage_5_predictions.loc[
            stage_5_predictions["Model"].isin(stage_5_1_models),
            "NSID",
        ]
    ).isdisjoint(test_ids),
}

input_audit = pd.DataFrame(
    {
        "Check": list(input_checks.keys()),
        "Passed": list(input_checks.values()),
    }
)

display(input_audit)

if not input_audit["Passed"].all():
    failed = input_audit.loc[
        ~input_audit["Passed"],
        "Check",
    ].tolist()
    raise AssertionError(
        "Stage 5.1 input audit failed: " + "; ".join(failed)
    )

print("\nStage 5.1 input audit passed.")


,Check,Passed
0,Stage 4 audit passed,True
1,Stage 5 audit passed,True
2,"Training sample contains 7,619 participants",True
3,Training identifiers are unique,True
4,Training and test identifiers do not overlap,True
5,Training data contain no test identifiers,True
6,Outer folds contain exactly the training ident...,True
7,All 69 predictors are registered,True
8,All 10 represented domains are available,True
9,LODO specification contains 10 domains,True



Stage 5.1 input audit passed.


## Part 2: Model and evaluation functions

The Stage 5-selected hyperparameters and imbalance setting are reused within each outer fold; no new hyperparameter search is performed for a reduced or augmented predictor set. Preprocessing is fitted on the relevant outer-training sample using only the predictors in the current specification.

The comparisons therefore measure performance change under the fixed Stage 5 model settings rather than the best attainable performance after retuning each predictor subset.

In [4]:
# 4: Define predictor roles and model outcome labels

numeric_predictors = predictor_roles.loc[
    predictor_roles["Preprocessing role"].eq("Numeric"),
    "Predictor",
].tolist()

categorical_predictors = predictor_roles.loc[
    predictor_roles["Preprocessing role"].eq("Categorical"),
    "Predictor",
].tolist()

assert len(numeric_predictors) + len(categorical_predictors) == 69
assert set(numeric_predictors).isdisjoint(categorical_predictors)

for predictor in numeric_predictors:
    training_data[predictor] = pd.to_numeric(
        training_data[predictor],
        errors="raise",
    )

expected_outcomes = {
    1: "Education",
    4: "Employment",
    5: "Apprenticeship or training",
    6: "Unemployment or inactivity (NEET)",
}

outcome_code_to_internal = {1: 0, 4: 1, 5: 2, 6: 3}
internal_to_outcome_code = {
    value: key
    for key, value in outcome_code_to_internal.items()
}
internal_to_outcome = {
    outcome_code_to_internal[code]: label
    for code, label in expected_outcomes.items()
}

training_data["model_outcome"] = (
    training_data["age18_outcome_code"]
    .map(outcome_code_to_internal)
    .astype(int)
)

class_order = [0, 1, 2, 3]

print(f"Numeric/ordinal predictors: {len(numeric_predictors)}")
print(f"Binary/nominal predictors: {len(categorical_predictors)}")


Numeric/ordinal predictors: 38
Binary/nominal predictors: 31


In [5]:
# 5: Define overall and destination-specific metrics

def overall_metrics(y_true, y_pred):
    return {
        "Macro F1": f1_score(
            y_true,
            y_pred,
            labels=class_order,
            average="macro",
            zero_division=0,
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Weighted F1": f1_score(
            y_true,
            y_pred,
            labels=class_order,
            average="weighted",
            zero_division=0,
        ),
    }


def class_metrics(y_true, y_pred):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=class_order,
            zero_division=0,
        )
    )

    rows = []
    for index, internal_code in enumerate(class_order):
        rows.append(
            {
                "Internal class": internal_code,
                "Outcome code": internal_to_outcome_code[
                    internal_code
                ],
                "Outcome": internal_to_outcome[
                    internal_code
                ],
                "Precision": precision[index],
                "Recall": recall[index],
                "F1": f1[index],
                "Support": int(support[index]),
            }
        )

    return rows


print("Evaluation functions defined.")


Evaluation functions defined.


In [6]:
# 6: Define subset-specific preprocessing and model fitting

stage_5_search_index = stage_5_search.set_index(
    "Stage 5 model label"
)

MODEL_RANDOM_STATE = int(
    json.loads(
        stage_5_search_index.loc[
            "RF",
            "Fixed parameters",
        ]
    )["random_state"]
)


def make_preprocessor(model_name, predictors):
    # Use only predictors included in the current domain specification.
    current_numeric = [
        predictor
        for predictor in predictors
        if predictor in numeric_predictors
    ]
    current_categorical = [
        predictor
        for predictor in predictors
        if predictor in categorical_predictors
    ]

    transformers = []

    if current_numeric:
        if model_name == "MLR":
            numeric_pipeline = Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            )
        else:
            numeric_pipeline = Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            )

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                current_numeric,
            )
        )

    if current_categorical:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        drop="if_binary",
                    ),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                current_categorical,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


def make_estimator(model_name, parameters, weighting_mode):
    if model_name == "MLR":
        return LogisticRegression(
            C=float(parameters["C"]),
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            class_weight=(
                "balanced"
                if weighting_mode == "Balanced"
                else None
            ),
        )

    if model_name == "RF":
        return RandomForestClassifier(
            **parameters,
            min_samples_split=2,
            criterion="gini",
            class_weight=(
                "balanced"
                if weighting_mode == "Balanced"
                else None
            ),
            n_jobs=-1,
            random_state=MODEL_RANDOM_STATE,
        )

    if model_name == "XGBoost":
        return XGBClassifier(
            **parameters,
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=MODEL_RANDOM_STATE,
            verbosity=0,
        )

    if model_name == "BRF":
        return BalancedRandomForestClassifier(
            **parameters,
            min_samples_split=2,
            criterion="gini",
            sampling_strategy="all",
            replacement=True,
            bootstrap=False,
            class_weight=None,
            n_jobs=-1,
            random_state=MODEL_RANDOM_STATE,
        )

    raise ValueError(f"Unknown model: {model_name}")


def make_model_pipeline(
    model_name,
    parameters,
    weighting_mode,
    predictors,
):
    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor(
                    model_name,
                    predictors,
                ),
            ),
            (
                "model",
                make_estimator(
                    model_name,
                    parameters,
                    weighting_mode,
                ),
            ),
        ]
    )


def fit_pipeline(
    pipeline,
    X,
    y,
    model_name,
    weighting_mode,
):
    if (
        model_name == "XGBoost"
        and weighting_mode == "Balanced"
    ):
        sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y,
        )
        pipeline.fit(
            X,
            y,
            model__sample_weight=sample_weights,
        )
    else:
        pipeline.fit(X, y)

    return pipeline


print("Subset-specific preprocessing and model functions defined.")


Subset-specific preprocessing and model functions defined.


In [7]:
# 7: Define outer-fold evaluation helpers

selection_index = (
    stage_5_selections.loc[
        stage_5_selections["Model"].isin(stage_5_1_models)
    ]
    .set_index(["Model", "Outer fold"])
)


def outer_fold_data(outer_fold):
    validation_ids = set(
        outer_assignments.loc[
            outer_assignments["outer_fold"].eq(
                outer_fold
            ),
            "NSID",
        ]
    )

    outer_training = training_data.loc[
        ~training_data["NSID"].isin(validation_ids)
    ].copy()

    outer_validation = training_data.loc[
        training_data["NSID"].isin(validation_ids)
    ].copy()

    return outer_training, outer_validation


def selected_stage_5_specification(model_name, outer_fold):
    row = selection_index.loc[(model_name, outer_fold)]

    return {
        "weighting_mode": row["Selected weighting mode"],
        "candidate_id": int(row["Selected candidate ID"]),
        "parameters": json.loads(row["Selected parameters"]),
    }


def evaluate_specification(
    specification,
    predictors,
    model_name,
    outer_fold,
):
    outer_training, outer_validation = outer_fold_data(
        outer_fold
    )

    selected = selected_stage_5_specification(
        model_name,
        outer_fold,
    )

    pipeline = make_model_pipeline(
        model_name,
        selected["parameters"],
        selected["weighting_mode"],
        predictors,
    )

    start_time = time.perf_counter()

    with warnings.catch_warnings():
        warnings.simplefilter("default")
        fit_pipeline(
            pipeline,
            outer_training[predictors],
            outer_training["model_outcome"],
            model_name,
            selected["weighting_mode"],
        )

    fit_seconds = time.perf_counter() - start_time

    prediction = pipeline.predict(
        outer_validation[predictors]
    )

    overall_row = {
        "Specification": specification,
        "Model": model_name,
        "Outer fold": outer_fold,
        "Predictors": len(predictors),
        "Selected weighting mode": selected["weighting_mode"],
        "Selected candidate ID": selected["candidate_id"],
        "Selected parameters": json.dumps(
            selected["parameters"],
            sort_keys=True,
        ),
        "Outer fit seconds": fit_seconds,
        **overall_metrics(
            outer_validation["model_outcome"],
            prediction,
        ),
    }

    class_rows = []
    for row in class_metrics(
        outer_validation["model_outcome"],
        prediction,
    ):
        class_rows.append(
            {
                "Specification": specification,
                "Model": model_name,
                "Outer fold": outer_fold,
                "Predictors": len(predictors),
                "Selected weighting mode": selected[
                    "weighting_mode"
                ],
                "Selected candidate ID": selected[
                    "candidate_id"
                ],
                **row,
            }
        )

    prediction_table = pd.DataFrame(
        {
            "NSID": outer_validation["NSID"].to_numpy(),
            "Predicted internal class": np.asarray(
                prediction,
                dtype=int,
            ),
        }
    )

    return overall_row, class_rows, prediction_table


def replace_model_rows(path, new_rows, model_name):
    new_rows = new_rows.copy()

    if path.exists():
        existing = pd.read_csv(
            path,
            dtype={"NSID": "string"},
            keep_default_na=False,
            na_values=[""],
        )
        if "Model" in existing.columns:
            existing = existing.loc[
                ~existing["Model"].eq(model_name)
            ]
        combined = pd.concat(
            [existing, new_rows],
            ignore_index=True,
        )
    else:
        combined = new_rows

    combined.to_csv(path, index=False)


print("Outer-fold evaluation helpers defined.")


Outer-fold evaluation helpers defined.


## Part 3: Full-model reproduction

Before any domain comparison, the Stage 5 full-model predictions are reproduced using the Stage 5-selected outer-fold specifications. This checks that Stage 5.1 uses the same preprocessing, weighting and estimator settings as Stage 5.


In [8]:
# 8: Reproduce the Stage 5 full-model outer predictions

full_overall_rows = []
full_class_rows = []
reproduction_rows = []

for model_name in stage_5_1_models:
    for outer_fold in range(1, 6):
        (
            overall_row,
            class_rows,
            reproduced_predictions,
        ) = evaluate_specification(
            specification="Full 69 predictors",
            predictors=predictor_columns,
            model_name=model_name,
            outer_fold=outer_fold,
        )

        full_overall_rows.append(overall_row)
        full_class_rows.extend(class_rows)

        stored = (
            stage_5_predictions.loc[
                stage_5_predictions["Model"].eq(model_name)
                & stage_5_predictions["Outer fold"].eq(
                    outer_fold
                ),
                ["NSID", "Predicted internal class"],
            ]
            .rename(
                columns={
                    "Predicted internal class":
                    "Stage 5 predicted internal class"
                }
            )
        )

        comparison = stored.merge(
            reproduced_predictions.rename(
                columns={
                    "Predicted internal class":
                    "Stage 5.1 predicted internal class"
                }
            ),
            on="NSID",
            how="inner",
            validate="one_to_one",
        )

        exact_matches = (
            comparison[
                "Stage 5 predicted internal class"
            ]
            .astype(int)
            .eq(
                comparison[
                    "Stage 5.1 predicted internal class"
                ].astype(int)
            )
            .sum()
        )

        reproduction_rows.append(
            {
                "Model": model_name,
                "Outer fold": outer_fold,
                "Compared participants": len(comparison),
                "Exact prediction matches": int(exact_matches),
                "Exact reproduction": (
                    len(comparison)
                    == len(reproduced_predictions)
                    == len(stored)
                    and exact_matches == len(comparison)
                ),
            }
        )

        print(
            f"{model_name} | outer fold {outer_fold}: "
            f"{exact_matches:,}/{len(comparison):,} "
            "predictions reproduced"
        )

full_overall = pd.DataFrame(full_overall_rows)
full_class = pd.DataFrame(full_class_rows)
reproduction_audit = pd.DataFrame(reproduction_rows)

reproduction_path = (
    stage_5_1_directory
    / "stage_5_1_full_model_reproduction_audit.csv"
)
reproduction_audit.to_csv(
    reproduction_path,
    index=False,
)

display(reproduction_audit)

if not reproduction_audit["Exact reproduction"].all():
    raise AssertionError(
        "Stage 5 full-model predictions were not reproduced exactly."
    )

print(f"\nSaved: {reproduction_path.relative_to(project_root)}")


MLR | outer fold 1: 1,524/1,524 predictions reproduced
MLR | outer fold 2: 1,524/1,524 predictions reproduced
MLR | outer fold 3: 1,524/1,524 predictions reproduced
MLR | outer fold 4: 1,524/1,524 predictions reproduced
MLR | outer fold 5: 1,523/1,523 predictions reproduced
RF | outer fold 1: 1,524/1,524 predictions reproduced
RF | outer fold 2: 1,524/1,524 predictions reproduced
RF | outer fold 3: 1,524/1,524 predictions reproduced
RF | outer fold 4: 1,524/1,524 predictions reproduced
RF | outer fold 5: 1,523/1,523 predictions reproduced
XGBoost | outer fold 1: 1,524/1,524 predictions reproduced
XGBoost | outer fold 2: 1,524/1,524 predictions reproduced
XGBoost | outer fold 3: 1,524/1,524 predictions reproduced
XGBoost | outer fold 4: 1,524/1,524 predictions reproduced
XGBoost | outer fold 5: 1,523/1,523 predictions reproduced
BRF | outer fold 1: 1,524/1,524 predictions reproduced
BRF | outer fold 2: 1,524/1,524 predictions reproduced
BRF | outer fold 3: 1,524/1,524 predictions reprod

,Model,Outer fold,Compared participants,Exact prediction matches,Exact reproduction
0,MLR,1,1524,1524,True
1,MLR,2,1524,1524,True
2,MLR,3,1524,1524,True
3,MLR,4,1524,1524,True
4,MLR,5,1523,1523,True
5,RF,1,1524,1524,True
6,RF,2,1524,1524,True
7,RF,3,1524,1524,True
8,RF,4,1524,1524,True
9,RF,5,1523,1523,True



Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_full_model_reproduction_audit.csv


## Part 4: Leave-one-domain-out analysis

Each of the 10 represented domains is omitted separately from the full predictor set.

A positive loss means performance was lower after omission:

**LODO loss = full-model performance − domain-omitted performance**


In [9]:
# 9: Run leave-one-domain-out comparisons

domain_predictors = (
    predictor_manifest
    .groupby("Predictor domain")["Predictor"]
    .apply(list)
    .to_dict()
)

lodo_overall_path = (
    stage_5_1_directory
    / "stage_5_1_lodo_outer_performance.csv"
)
lodo_class_path = (
    stage_5_1_directory
    / "stage_5_1_lodo_outer_class_performance.csv"
)

omission_domains = (
    domain_omission_spec["Predictor domain"].tolist()
)

for model_name in stage_5_1_models:
    model_overall_rows = []
    model_class_rows = []

    for omitted_domain in omission_domains:
        omitted_predictors = set(
            domain_predictors[omitted_domain]
        )
        retained_predictors = [
            predictor
            for predictor in predictor_columns
            if predictor not in omitted_predictors
        ]

        domain_fold_losses = []

        for outer_fold in range(1, 6):
            overall_row, class_rows, _ = evaluate_specification(
                specification=(
                    f"Full minus {omitted_domain}"
                ),
                predictors=retained_predictors,
                model_name=model_name,
                outer_fold=outer_fold,
            )

            full_reference = full_overall.loc[
                full_overall["Model"].eq(model_name)
                & full_overall["Outer fold"].eq(
                    outer_fold
                )
            ].iloc[0]

            overall_row["Omitted domain"] = omitted_domain
            overall_row["Full macro F1"] = full_reference[
                "Macro F1"
            ]
            overall_row["Macro F1 loss"] = (
                full_reference["Macro F1"]
                - overall_row["Macro F1"]
            )
            overall_row["Full balanced accuracy"] = (
                full_reference["Balanced accuracy"]
            )
            overall_row["Balanced accuracy loss"] = (
                full_reference["Balanced accuracy"]
                - overall_row["Balanced accuracy"]
            )

            model_overall_rows.append(overall_row)
            domain_fold_losses.append(
                overall_row["Macro F1 loss"]
            )

            full_class_reference = full_class.loc[
                full_class["Model"].eq(model_name)
                & full_class["Outer fold"].eq(
                    outer_fold
                )
            ].set_index("Outcome")

            for row in class_rows:
                reference = full_class_reference.loc[
                    row["Outcome"]
                ]

                row["Omitted domain"] = omitted_domain
                row["Full precision"] = reference[
                    "Precision"
                ]
                row["Precision loss"] = (
                    reference["Precision"]
                    - row["Precision"]
                )
                row["Full recall"] = reference["Recall"]
                row["Recall loss"] = (
                    reference["Recall"]
                    - row["Recall"]
                )
                row["Full F1"] = reference["F1"]
                row["F1 loss"] = (
                    reference["F1"]
                    - row["F1"]
                )

                model_class_rows.append(row)

        print(
            f"{model_name} | omitted {omitted_domain}: "
            f"mean macro-F1 loss = "
            f"{np.mean(domain_fold_losses):.4f}"
        )

    replace_model_rows(
        lodo_overall_path,
        pd.DataFrame(model_overall_rows),
        model_name,
    )
    replace_model_rows(
        lodo_class_path,
        pd.DataFrame(model_class_rows),
        model_name,
    )

lodo_outer = pd.read_csv(
    lodo_overall_path,
    keep_default_na=False,
    na_values=[""],
)
lodo_class_outer = pd.read_csv(
    lodo_class_path,
    keep_default_na=False,
    na_values=[""],
)

print(f"\nSaved: {lodo_overall_path.relative_to(project_root)}")
print(f"Saved: {lodo_class_path.relative_to(project_root)}")


MLR | omitted Demographic background: mean macro-F1 loss = 0.0057
MLR | omitted Educational aspirations and post-16 plans: mean macro-F1 loss = 0.0073
MLR | omitted Experiences and behaviours: mean macro-F1 loss = -0.0012
MLR | omitted Family socioeconomic background: mean macro-F1 loss = -0.0041
MLR | omitted Parental attitudes, support and engagement: mean macro-F1 loss = -0.0012
MLR | omitted Post-16 social influences and guidance: mean macro-F1 loss = -0.0006
MLR | omitted Psychosocial characteristics: mean macro-F1 loss = 0.0001
MLR | omitted SEN, disability and health: mean macro-F1 loss = 0.0047
MLR | omitted School and local context: mean macro-F1 loss = 0.0024
MLR | omitted School experiences and engagement: mean macro-F1 loss = -0.0005
RF | omitted Demographic background: mean macro-F1 loss = 0.0046
RF | omitted Educational aspirations and post-16 plans: mean macro-F1 loss = 0.0210
RF | omitted Experiences and behaviours: mean macro-F1 loss = 0.0087
RF | omitted Family socioe

In [10]:
# 10: Summarise LODO performance loss across outer folds

lodo_summary = (
    lodo_outer
    .groupby(
        ["Model", "Omitted domain"],
        as_index=False,
    )
    .agg(
        Outer_folds=("Outer fold", "nunique"),
        Mean_macro_F1_loss=("Macro F1 loss", "mean"),
        SD_macro_F1_loss=("Macro F1 loss", "std"),
        Minimum_macro_F1_loss=("Macro F1 loss", "min"),
        Maximum_macro_F1_loss=("Macro F1 loss", "max"),
        Positive_macro_F1_loss_folds=(
            "Macro F1 loss",
            lambda values: int((values > 0).sum()),
        ),
        Mean_balanced_accuracy_loss=(
            "Balanced accuracy loss",
            "mean",
        ),
        SD_balanced_accuracy_loss=(
            "Balanced accuracy loss",
            "std",
        ),
    )
    .sort_values(
        ["Model", "Mean_macro_F1_loss"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

assert lodo_summary["Outer_folds"].eq(5).all()

lodo_summary_path = (
    stage_5_1_directory
    / "stage_5_1_lodo_summary.csv"
)
lodo_summary.to_csv(
    lodo_summary_path,
    index=False,
)

display(lodo_summary.round(6))
print(
    f"Saved: "
    f"{lodo_summary_path.relative_to(project_root)}"
)


,Model,Omitted domain,Outer_folds,Mean_macro_F1_loss,SD_macro_F1_loss,Minimum_macro_F1_loss,Maximum_macro_F1_loss,Positive_macro_F1_loss_folds,Mean_balanced_accuracy_loss,SD_balanced_accuracy_loss
0,BRF,Educational aspirations and post-16 plans,5,0.016040,0.014944,-0.007370,0.028381,4,0.013837,0.013884
1,BRF,Family socioeconomic background,5,0.006120,0.007038,-0.004217,0.014106,4,0.004881,0.007520
2,BRF,Experiences and behaviours,5,0.005808,0.004315,-0.000946,0.009971,4,0.004789,0.005537
3,BRF,Demographic background,5,0.003833,0.015017,-0.013048,0.015973,3,0.002667,0.012944
4,BRF,School experiences and engagement,5,0.002301,0.012913,-0.016958,0.015740,3,0.000998,0.012707
5,BRF,"Parental attitudes, support and engagement",5,0.001290,0.010804,-0.014067,0.015159,3,-0.000009,0.009828
6,BRF,"SEN, disability and health",5,0.000203,0.010653,-0.011396,0.011496,2,-0.000797,0.010345
7,BRF,School and local context,5,-0.000125,0.006440,-0.006383,0.008753,2,-0.001108,0.006886
8,BRF,Psychosocial characteristics,5,-0.000779,0.007133,-0.008947,0.009922,2,-0.000800,0.009272
9,BRF,Post-16 social influences and guidance,5,-0.002559,0.008037,-0.010083,0.007183,2,-0.001571,0.007214


Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_lodo_summary.csv


In [11]:
# 11: Summarise destination-specific LODO changes

lodo_class_summary = (
    lodo_class_outer
    .groupby(
        ["Model", "Omitted domain", "Outcome"],
        as_index=False,
    )
    .agg(
        Outer_folds=("Outer fold", "nunique"),
        Mean_precision_loss=("Precision loss", "mean"),
        SD_precision_loss=("Precision loss", "std"),
        Mean_recall_loss=("Recall loss", "mean"),
        SD_recall_loss=("Recall loss", "std"),
        Mean_F1_loss=("F1 loss", "mean"),
        SD_F1_loss=("F1 loss", "std"),
    )
    .sort_values(
        ["Model", "Omitted domain", "Outcome"]
    )
    .reset_index(drop=True)
)

assert lodo_class_summary["Outer_folds"].eq(5).all()

lodo_class_summary_path = (
    stage_5_1_directory
    / "stage_5_1_lodo_class_summary.csv"
)
lodo_class_summary.to_csv(
    lodo_class_summary_path,
    index=False,
)

display(lodo_class_summary.round(6))
print(
    f"Saved: "
    f"{lodo_class_summary_path.relative_to(project_root)}"
)


,Model,Omitted domain,Outcome,Outer_folds,Mean_precision_loss,SD_precision_loss,Mean_recall_loss,SD_recall_loss,Mean_F1_loss,SD_F1_loss
0,BRF,Demographic background,Apprenticeship or training,5,-0.000535,0.023317,0.002352,0.015610,0.000064,0.020040
1,BRF,Demographic background,Education,5,0.008547,0.004927,-0.002775,0.007047,0.003141,0.003792
2,BRF,Demographic background,Employment,5,-0.002206,0.025318,0.024081,0.034237,0.013563,0.030477
3,BRF,Demographic background,Unemployment or inactivity (NEET),5,0.008525,0.024347,-0.012990,0.010372,-0.001434,0.017251
4,BRF,Educational aspirations and post-16 plans,Apprenticeship or training,5,0.023516,0.027328,0.021773,0.044455,0.023527,0.034031
...,...,...,...,...,...,...,...,...,...,...
155,XGBoost,School and local context,Unemployment or inactivity (NEET),5,0.004677,0.015837,0.002005,0.010364,0.003302,0.012531
156,XGBoost,School experiences and engagement,Apprenticeship or training,5,0.029469,0.029219,-0.012106,0.022425,0.005112,0.023226
157,XGBoost,School experiences and engagement,Education,5,-0.001033,0.003525,0.008330,0.006221,0.003722,0.004739
158,XGBoost,School experiences and engagement,Employment,5,-0.001297,0.006744,0.008040,0.016594,0.003131,0.010336


Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_lodo_class_summary.csv


## Part 5: Common-baseline separate addition

The common baseline is fitted once within each model × outer-fold combination. Each of the six additional domains is then added separately to that same baseline.

A positive gain means performance was higher after adding the domain:

**Addition gain = augmented performance − common-baseline performance**

In [12]:
# 12: Build the common baseline and six separate additions

baseline_row = domain_addition_spec.loc[
    domain_addition_spec["Analysis"].eq(
        "Common-baseline reference"
    )
].iloc[0]

baseline_domains = [
    value.strip()
    for value in baseline_row["Included domains"].split("|")
]

addition_rows = domain_addition_spec.loc[
    domain_addition_spec["Analysis"].eq(
        "Separate domain addition"
    )
].copy()

baseline_predictors = []
for domain in baseline_domains:
    baseline_predictors.extend(
        domain_predictors[domain]
    )

baseline_predictors = [
    predictor
    for predictor in predictor_columns
    if predictor in set(baseline_predictors)
]

addition_domains = addition_rows["Added domain"].tolist()

assert len(baseline_predictors) == int(
    baseline_row["Predictors"]
)
assert len(addition_domains) == 6
assert set(baseline_domains).isdisjoint(addition_domains)
assert set(baseline_domains) | set(addition_domains) == set(domain_predictors)
assert len(baseline_predictors) == 23

print("Common baseline domains:")
for domain in baseline_domains:
    print(f"  - {domain}")

print(
    f"\nCommon baseline predictors: "
    f"{len(baseline_predictors)}"
)
print("\nSeparate additions:")
for domain in addition_domains:
    print(
        f"  - {domain} "
        f"({len(domain_predictors[domain])} predictors)"
    )


Common baseline domains:
  - Demographic background
  - Family socioeconomic background
  - SEN, disability and health
  - School and local context

Common baseline predictors: 23

Separate additions:
  - Educational aspirations and post-16 plans (2 predictors)
  - School experiences and engagement (9 predictors)
  - Psychosocial characteristics (2 predictors)
  - Experiences and behaviours (7 predictors)
  - Parental attitudes, support and engagement (16 predictors)
  - Post-16 social influences and guidance (10 predictors)


In [13]:
# 13: Run common-baseline separate additions

baseline_overall_path = (
    stage_5_1_directory
    / "stage_5_1_common_baseline_outer_performance.csv"
)
baseline_class_path = (
    stage_5_1_directory
    / "stage_5_1_common_baseline_outer_class_performance.csv"
)
addition_overall_path = (
    stage_5_1_directory
    / "stage_5_1_addition_outer_performance.csv"
)
addition_class_path = (
    stage_5_1_directory
    / "stage_5_1_addition_outer_class_performance.csv"
)

for model_name in stage_5_1_models:
    model_baseline_overall = []
    model_baseline_class = []
    model_addition_overall = []
    model_addition_class = []

    baseline_overall_by_fold = {}
    baseline_class_by_fold = {}

    # Fit the common baseline once per outer fold.
    for outer_fold in range(1, 6):
        (
            baseline_overall_row,
            baseline_class_rows,
            _,
        ) = evaluate_specification(
            specification="Common baseline",
            predictors=baseline_predictors,
            model_name=model_name,
            outer_fold=outer_fold,
        )

        model_baseline_overall.append(
            baseline_overall_row
        )
        model_baseline_class.extend(
            baseline_class_rows
        )

        baseline_overall_by_fold[
            outer_fold
        ] = baseline_overall_row

        baseline_class_by_fold[
            outer_fold
        ] = pd.DataFrame(
            baseline_class_rows
        ).set_index("Outcome")

    # Fit each added domain against the same baseline.
    for added_domain in addition_domains:
        augmented_predictor_set = set(
            baseline_predictors
            + domain_predictors[added_domain]
        )
        augmented_predictors = [
            predictor
            for predictor in predictor_columns
            if predictor in augmented_predictor_set
        ]

        domain_fold_gains = []

        for outer_fold in range(1, 6):
            overall_row, class_rows, _ = evaluate_specification(
                specification=(
                    f"Common baseline + {added_domain}"
                ),
                predictors=augmented_predictors,
                model_name=model_name,
                outer_fold=outer_fold,
            )

            baseline_reference = (
                baseline_overall_by_fold[outer_fold]
            )

            overall_row["Added domain"] = added_domain
            overall_row["Baseline macro F1"] = (
                baseline_reference["Macro F1"]
            )
            overall_row["Macro F1 gain"] = (
                overall_row["Macro F1"]
                - baseline_reference["Macro F1"]
            )
            overall_row[
                "Baseline balanced accuracy"
            ] = baseline_reference[
                "Balanced accuracy"
            ]
            overall_row[
                "Balanced accuracy gain"
            ] = (
                overall_row["Balanced accuracy"]
                - baseline_reference[
                    "Balanced accuracy"
                ]
            )

            model_addition_overall.append(overall_row)
            domain_fold_gains.append(
                overall_row["Macro F1 gain"]
            )

            class_reference = (
                baseline_class_by_fold[outer_fold]
            )

            for row in class_rows:
                reference = class_reference.loc[
                    row["Outcome"]
                ]

                row["Added domain"] = added_domain
                row["Baseline precision"] = reference[
                    "Precision"
                ]
                row["Precision gain"] = (
                    row["Precision"]
                    - reference["Precision"]
                )
                row["Baseline recall"] = reference[
                    "Recall"
                ]
                row["Recall gain"] = (
                    row["Recall"]
                    - reference["Recall"]
                )
                row["Baseline F1"] = reference["F1"]
                row["F1 gain"] = (
                    row["F1"]
                    - reference["F1"]
                )

                model_addition_class.append(row)

        print(
            f"{model_name} | added {added_domain}: "
            f"mean macro-F1 gain = "
            f"{np.mean(domain_fold_gains):.4f}"
        )

    replace_model_rows(
        baseline_overall_path,
        pd.DataFrame(model_baseline_overall),
        model_name,
    )
    replace_model_rows(
        baseline_class_path,
        pd.DataFrame(model_baseline_class),
        model_name,
    )
    replace_model_rows(
        addition_overall_path,
        pd.DataFrame(model_addition_overall),
        model_name,
    )
    replace_model_rows(
        addition_class_path,
        pd.DataFrame(model_addition_class),
        model_name,
    )

baseline_outer = pd.read_csv(
    baseline_overall_path,
    keep_default_na=False,
    na_values=[""],
)
baseline_class_outer = pd.read_csv(
    baseline_class_path,
    keep_default_na=False,
    na_values=[""],
)
addition_outer = pd.read_csv(
    addition_overall_path,
    keep_default_na=False,
    na_values=[""],
)
addition_class_outer = pd.read_csv(
    addition_class_path,
    keep_default_na=False,
    na_values=[""],
)

print(f"\nSaved: {baseline_overall_path.relative_to(project_root)}")
print(f"Saved: {baseline_class_path.relative_to(project_root)}")
print(f"Saved: {addition_overall_path.relative_to(project_root)}")
print(f"Saved: {addition_class_path.relative_to(project_root)}")


MLR | added Educational aspirations and post-16 plans: mean macro-F1 gain = 0.0482
MLR | added School experiences and engagement: mean macro-F1 gain = 0.0189
MLR | added Psychosocial characteristics: mean macro-F1 gain = 0.0148
MLR | added Experiences and behaviours: mean macro-F1 gain = 0.0087
MLR | added Parental attitudes, support and engagement: mean macro-F1 gain = 0.0277
MLR | added Post-16 social influences and guidance: mean macro-F1 gain = 0.0328
RF | added Educational aspirations and post-16 plans: mean macro-F1 gain = 0.0562
RF | added School experiences and engagement: mean macro-F1 gain = 0.0261
RF | added Psychosocial characteristics: mean macro-F1 gain = 0.0166
RF | added Experiences and behaviours: mean macro-F1 gain = 0.0060
RF | added Parental attitudes, support and engagement: mean macro-F1 gain = 0.0398
RF | added Post-16 social influences and guidance: mean macro-F1 gain = 0.0344
XGBoost | added Educational aspirations and post-16 plans: mean macro-F1 gain = 0.0541

In [14]:
# 14: Summarise separate domain-addition gains

addition_summary = (
    addition_outer
    .groupby(
        ["Model", "Added domain"],
        as_index=False,
    )
    .agg(
        Outer_folds=("Outer fold", "nunique"),
        Mean_macro_F1_gain=("Macro F1 gain", "mean"),
        SD_macro_F1_gain=("Macro F1 gain", "std"),
        Minimum_macro_F1_gain=("Macro F1 gain", "min"),
        Maximum_macro_F1_gain=("Macro F1 gain", "max"),
        Positive_macro_F1_gain_folds=(
            "Macro F1 gain",
            lambda values: int((values > 0).sum()),
        ),
        Mean_balanced_accuracy_gain=(
            "Balanced accuracy gain",
            "mean",
        ),
        SD_balanced_accuracy_gain=(
            "Balanced accuracy gain",
            "std",
        ),
    )
    .sort_values(
        ["Model", "Mean_macro_F1_gain"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

assert addition_summary["Outer_folds"].eq(5).all()

addition_summary_path = (
    stage_5_1_directory
    / "stage_5_1_addition_summary.csv"
)
addition_summary.to_csv(
    addition_summary_path,
    index=False,
)

display(addition_summary.round(6))
print(
    f"Saved: "
    f"{addition_summary_path.relative_to(project_root)}"
)


,Model,Added domain,Outer_folds,Mean_macro_F1_gain,SD_macro_F1_gain,Minimum_macro_F1_gain,Maximum_macro_F1_gain,Positive_macro_F1_gain_folds,Mean_balanced_accuracy_gain,SD_balanced_accuracy_gain
0,BRF,Educational aspirations and post-16 plans,5,0.063009,0.008800,0.049121,0.072070,5,0.051078,0.009241
1,BRF,"Parental attitudes, support and engagement",5,0.038392,0.017061,0.015004,0.059339,5,0.027906,0.019145
2,BRF,Post-16 social influences and guidance,5,0.035560,0.008066,0.026065,0.042937,5,0.028695,0.008734
3,BRF,School experiences and engagement,5,0.026025,0.011440,0.011326,0.042318,5,0.019541,0.013558
4,BRF,Psychosocial characteristics,5,0.017976,0.015338,0.003688,0.042508,5,0.014291,0.017076
5,BRF,Experiences and behaviours,5,0.008999,0.007430,-0.002312,0.016879,4,-0.000467,0.009291
6,MLR,Educational aspirations and post-16 plans,5,0.048154,0.010431,0.030990,0.055842,5,0.041400,0.012917
7,MLR,Post-16 social influences and guidance,5,0.032754,0.008244,0.020210,0.039318,5,0.031194,0.016433
8,MLR,"Parental attitudes, support and engagement",5,0.027718,0.005294,0.019075,0.033487,5,0.023936,0.007637
9,MLR,School experiences and engagement,5,0.018925,0.009553,0.004500,0.027731,5,0.019168,0.013049


Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_addition_summary.csv


In [15]:
# 15: Summarise destination-specific addition gains

addition_class_summary = (
    addition_class_outer
    .groupby(
        ["Model", "Added domain", "Outcome"],
        as_index=False,
    )
    .agg(
        Outer_folds=("Outer fold", "nunique"),
        Mean_precision_gain=("Precision gain", "mean"),
        SD_precision_gain=("Precision gain", "std"),
        Mean_recall_gain=("Recall gain", "mean"),
        SD_recall_gain=("Recall gain", "std"),
        Mean_F1_gain=("F1 gain", "mean"),
        SD_F1_gain=("F1 gain", "std"),
    )
    .sort_values(
        ["Model", "Added domain", "Outcome"]
    )
    .reset_index(drop=True)
)

assert addition_class_summary["Outer_folds"].eq(5).all()

addition_class_summary_path = (
    stage_5_1_directory
    / "stage_5_1_addition_class_summary.csv"
)
addition_class_summary.to_csv(
    addition_class_summary_path,
    index=False,
)

display(addition_class_summary.round(6))
print(
    f"Saved: "
    f"{addition_class_summary_path.relative_to(project_root)}"
)


,Model,Added domain,Outcome,Outer_folds,Mean_precision_gain,SD_precision_gain,Mean_recall_gain,SD_recall_gain,Mean_F1_gain,SD_F1_gain
0,BRF,Educational aspirations and post-16 plans,Apprenticeship or training,5,0.083057,0.022959,0.007315,0.049289,0.077032,0.025329
1,BRF,Educational aspirations and post-16 plans,Education,5,0.017259,0.008899,0.161837,0.028063,0.100498,0.018545
2,BRF,Educational aspirations and post-16 plans,Employment,5,0.051288,0.021222,0.011174,0.031069,0.029095,0.021609
3,BRF,Educational aspirations and post-16 plans,Unemployment or inactivity (NEET),5,0.057438,0.004576,0.023985,0.008963,0.045411,0.002450
4,BRF,Experiences and behaviours,Apprenticeship or training,5,-0.000161,0.009289,-0.057659,0.021411,-0.009730,0.013833
...,...,...,...,...,...,...,...,...,...,...
91,XGBoost,Psychosocial characteristics,Unemployment or inactivity (NEET),5,0.006406,0.011182,0.007020,0.020461,0.006852,0.013941
92,XGBoost,School experiences and engagement,Apprenticeship or training,5,0.015829,0.006911,-0.062593,0.028861,0.001927,0.009429
93,XGBoost,School experiences and engagement,Education,5,0.012918,0.009916,0.072704,0.012087,0.050443,0.006405
94,XGBoost,School experiences and engagement,Employment,5,0.019005,0.018631,0.039270,0.023919,0.029781,0.019087


Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_addition_class_summary.csv


## Part 6: Integrated domain comparison

The six added domains also have LODO results. The two estimates are shown together because they answer different conditional questions: addition measures change from the common baseline, whereas LODO measures change from the full predictor set.

The estimates are not interchangeable importance scores. Domain sizes also differ, so the changes are not normalised per predictor.

In [16]:
# 16: Combine LODO and addition summaries for the six added domains

integrated_domain_summary = (
    addition_summary.merge(
        lodo_summary.rename(
            columns={
                "Omitted domain": "Added domain",
            }
        )[
            [
                "Model",
                "Added domain",
                "Mean_macro_F1_loss",
                "SD_macro_F1_loss",
                "Mean_balanced_accuracy_loss",
                "SD_balanced_accuracy_loss",
            ]
        ],
        on=["Model", "Added domain"],
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "Added domain": "Predictor domain",
        }
    )
)

integrated_path = (
    stage_5_1_directory
    / "stage_5_1_integrated_domain_summary.csv"
)
integrated_domain_summary.to_csv(
    integrated_path,
    index=False,
)

display(integrated_domain_summary.round(6))
print(
    f"Saved: {integrated_path.relative_to(project_root)}"
)


,Model,Predictor domain,Outer_folds,Mean_macro_F1_gain,SD_macro_F1_gain,Minimum_macro_F1_gain,Maximum_macro_F1_gain,Positive_macro_F1_gain_folds,Mean_balanced_accuracy_gain,SD_balanced_accuracy_gain,Mean_macro_F1_loss,SD_macro_F1_loss,Mean_balanced_accuracy_loss,SD_balanced_accuracy_loss
0,BRF,Educational aspirations and post-16 plans,5,0.063009,0.008800,0.049121,0.072070,5,0.051078,0.009241,0.016040,0.014944,0.013837,0.013884
1,BRF,"Parental attitudes, support and engagement",5,0.038392,0.017061,0.015004,0.059339,5,0.027906,0.019145,0.001290,0.010804,-0.000009,0.009828
2,BRF,Post-16 social influences and guidance,5,0.035560,0.008066,0.026065,0.042937,5,0.028695,0.008734,-0.002559,0.008037,-0.001571,0.007214
3,BRF,School experiences and engagement,5,0.026025,0.011440,0.011326,0.042318,5,0.019541,0.013558,0.002301,0.012913,0.000998,0.012707
4,BRF,Psychosocial characteristics,5,0.017976,0.015338,0.003688,0.042508,5,0.014291,0.017076,-0.000779,0.007133,-0.000800,0.009272
5,BRF,Experiences and behaviours,5,0.008999,0.007430,-0.002312,0.016879,4,-0.000467,0.009291,0.005808,0.004315,0.004789,0.005537
6,MLR,Educational aspirations and post-16 plans,5,0.048154,0.010431,0.030990,0.055842,5,0.041400,0.012917,0.007311,0.007603,0.000738,0.011976
7,MLR,Post-16 social influences and guidance,5,0.032754,0.008244,0.020210,0.039318,5,0.031194,0.016433,-0.000609,0.003680,-0.001807,0.005606
8,MLR,"Parental attitudes, support and engagement",5,0.027718,0.005294,0.019075,0.033487,5,0.023936,0.007637,-0.001163,0.007132,-0.001911,0.011143
9,MLR,School experiences and engagement,5,0.018925,0.009553,0.004500,0.027731,5,0.019168,0.013049,-0.000525,0.001863,0.000122,0.002558


Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_integrated_domain_summary.csv


## Part 7: Completion audit

In [17]:
# 17: Final Stage 5.1 audit and output manifest

expected_lodo_overall_rows = 4 * 5 * 10
expected_lodo_class_rows = expected_lodo_overall_rows * 4
expected_baseline_overall_rows = 4 * 5
expected_baseline_class_rows = expected_baseline_overall_rows * 4
expected_addition_overall_rows = 4 * 5 * 6
expected_addition_class_rows = expected_addition_overall_rows * 4

final_checks = {
    "Stage 5 full-model predictions were reproduced exactly": (
        len(reproduction_audit) == 20
        and reproduction_audit["Exact reproduction"].all()
    ),
    "LODO contains 200 model-fold-domain rows": (
        len(lodo_outer) == expected_lodo_overall_rows
    ),
    "LODO contains 800 destination-specific rows": (
        len(lodo_class_outer) == expected_lodo_class_rows
    ),
    "Every LODO model-domain comparison has five outer folds": (
        lodo_outer
        .groupby(["Model", "Omitted domain"])["Outer fold"]
        .nunique()
        .eq(5)
        .all()
    ),
    "Common baseline contains 20 model-fold rows": (
        len(baseline_outer) == expected_baseline_overall_rows
    ),
    "Common baseline contains 80 destination-specific rows": (
        len(baseline_class_outer) == expected_baseline_class_rows
    ),
    "Separate additions contain 120 model-fold-domain rows": (
        len(addition_outer) == expected_addition_overall_rows
    ),
    "Separate additions contain 480 destination-specific rows": (
        len(addition_class_outer) == expected_addition_class_rows
    ),
    "Every added-domain comparison has five outer folds": (
        addition_outer
        .groupby(["Model", "Added domain"])["Outer fold"]
        .nunique()
        .eq(5)
        .all()
    ),
    "MLR, RF, XGBoost and BRF are analysed": (
        set(lodo_outer["Model"])
        == set(addition_outer["Model"])
        == set(stage_5_1_models)
    ),
    "All 10 represented domains are covered by LODO": (
        set(lodo_outer["Omitted domain"])
        == set(omission_domains)
    ),
    "All six separate addition domains are covered": (
        set(addition_outer["Added domain"])
        == set(addition_domains)
    ),
}

final_audit = pd.DataFrame(
    {
        "Check": list(final_checks.keys()),
        "Passed": list(final_checks.values()),
    }
)

final_audit_path = (
    stage_5_1_directory
    / "stage_5_1_final_audit.csv"
)
final_audit.to_csv(
    final_audit_path,
    index=False,
)

display(final_audit)

if not final_audit["Passed"].all():
    failed = final_audit.loc[
        ~final_audit["Passed"],
        "Check",
    ].tolist()
    raise AssertionError(
        "Stage 5.1 final audit failed: "
        + "; ".join(failed)
    )

output_files = [
    "stage_5_1_full_model_reproduction_audit.csv",
    "stage_5_1_lodo_outer_performance.csv",
    "stage_5_1_lodo_outer_class_performance.csv",
    "stage_5_1_lodo_summary.csv",
    "stage_5_1_lodo_class_summary.csv",
    "stage_5_1_common_baseline_outer_performance.csv",
    "stage_5_1_common_baseline_outer_class_performance.csv",
    "stage_5_1_addition_outer_performance.csv",
    "stage_5_1_addition_outer_class_performance.csv",
    "stage_5_1_addition_summary.csv",
    "stage_5_1_addition_class_summary.csv",
    "stage_5_1_integrated_domain_summary.csv",
    "stage_5_1_final_audit.csv",
]

output_manifest = pd.DataFrame(
    {
        "File": output_files,
        "Exists": [
            (stage_5_1_directory / name).is_file()
            for name in output_files
        ],
    }
)
output_manifest["Path"] = output_manifest["File"].map(
    lambda name: str(
        (stage_5_1_directory / name)
        .relative_to(project_root)
    )
)

output_manifest_path = (
    stage_5_1_directory
    / "stage_5_1_output_manifest.csv"
)
output_manifest.to_csv(
    output_manifest_path,
    index=False,
)

print("\nStage 5.1 predictor-domain contribution is complete.")
print(
    f"Saved: "
    f"{final_audit_path.relative_to(project_root)}"
)
print(
    f"Saved: "
    f"{output_manifest_path.relative_to(project_root)}"
)


,Check,Passed
0,Stage 5 full-model predictions were reproduced...,True
1,LODO contains 200 model-fold-domain rows,True
2,LODO contains 800 destination-specific rows,True
3,Every LODO model-domain comparison has five ou...,True
4,Common baseline contains 20 model-fold rows,True
5,Common baseline contains 80 destination-specif...,True
6,Separate additions contain 120 model-fold-doma...,True
7,Separate additions contain 480 destination-spe...,True
8,Every added-domain comparison has five outer f...,True
9,"MLR, RF, XGBoost and BRF are analysed",True



Stage 5.1 predictor-domain contribution is complete.
Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_final_audit.csv
Saved: data_derived\stage_5_1_predictor_domain_contribution\stage_5_1_output_manifest.csv


## Stage 5.1 summary

All 20 model × outer-fold full-model prediction sets were reproduced exactly before the domain comparisons.

In LODO, educational aspirations and post-16 plans produced the largest mean macro-F1 loss for RF (0.0210) and XGBoost (0.0224), with positive losses in all five folds for both models. Mean losses for the same domain were 0.0073 for MLR and 0.0160 for BRF.

When the six additional domains were added separately to the common 23-predictor baseline, educational aspirations and post-16 plans produced the largest mean macro-F1 gain for every model: 0.0482 for MLR, 0.0562 for RF, 0.0541 for XGBoost and 0.0630 for BRF. These gains were positive in all five outer folds. Parental attitudes, support and engagement and post-16 social influences and guidance also produced positive mean gains across all four models.

LODO and separate addition are conditional comparisons under different predictor contexts. Small or negative LODO changes do not establish that a domain is uninformative or harmful, and fold counts are descriptive rather than significance tests. The results do not provide causal effects or a fixed ranking of substantive importance.